# 2.4 ter — Méthode apprise, APP-v3 : un autre modèle pré-entraîné

**Question.** Avec la même recette qu'APP-v2, un modèle pré-entraîné plus spécialisé que BioBERT fait-il mieux ?

**Ce qui change par rapport à APP-v2 : seulement le modèle.** Données, fenêtres, décodage, taux 5e-5 (retenu en
APP-v2), warmup 10 %, 10 époques évaluées une à une, graines 42 et 43, perte non pondérée.

| Modèle | Pré-entraînement | Casse |
|---|---|---|
| `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` (PubMedBERT) | PubMed, vocabulaire propre au biomédical | minuscules |
| `michiyasunaga/BioLinkBERT-base` | PubMed + liens de citation entre articles | minuscules |

Les deux ont l'architecture BERT (même classe `BertForTokenClassification`, même `BertTokenizer`). Ils mettent tout
en minuscules : l'indice de casse (« ALL » leucémie / « all ») est perdu.

**Référence BioBERT** : les runs APP-v2 au taux 5e-5 (graines 42 et 43), même recette, lus dans
`apprise_v2_reglage_validation.csv`.

**Règles fixées avant de lancer** :
1. pour chaque modèle, nombre d'époques choisi comme en APP-v2 (meilleure moyenne des deux graines ; à moins de
   0,005, le moins d'époques) ;
2. un nouveau modèle ne remplace BioBERT que si sa moyenne dépasse celle de BioBERT (0,892, 4 époques) de **plus de
   0,005** ; sinon APP-v2 reste la méthode apprise figée.

**Exécution** : Colab GPU T4, environ 30 minutes. Fichiers `apprise_v3_*`, modèle dans `modeles/apprise_v3`.

In [1]:
import os, sys, json, time, shutil, math
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RACINE = Path("/content/drive/MyDrive/projet_nlp_sante")
    DOSSIER_CHECKPOINTS = Path("/content/checkpoints")   # disque local de Colab, pas le Drive
    # Mise à jour du projet sur le Drive : si `MyDrive/maj_drive.zip` existe (archive préparée sur le serveur),
    # ses fichiers remplacent ceux du dossier du projet, puis l'archive est renommée pour ne pas être réappliquée.
    import zipfile
    MAJ = Path("/content/drive/MyDrive/maj_drive.zip")
    if MAJ.exists():
        with zipfile.ZipFile(MAJ) as z:
            z.extractall(RACINE)
            print(f"Drive mis à jour : {len(z.namelist())} fichiers depuis {MAJ.name}")
        MAJ.rename(MAJ.with_name("maj_drive_appliquee.zip"))
else:
    RACINE = Path.cwd().parent
    DOSSIER_CHECKPOINTS = RACINE / "modeles" / "checkpoints"
sys.path.insert(0, str(RACINE))

MODE_RAPIDE = os.environ.get("PROJET_MODE_RAPIDE") == "1"
print("Colab :", EN_COLAB, "| racine :", RACINE, "| mode rapide :", MODE_RAPIDE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Colab : True | racine : /content/drive/MyDrive/projet_nlp_sante | mode rapide : False


In [2]:
import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (BertTokenizer, AutoModelForTokenClassification, BertForTokenClassification,
                          TrainingArguments, DataCollatorForTokenClassification, set_seed)

from tals import erreurs
from src.cdr import charger_jsonl, mentions_reference
from src.apprise import (LABELS, LABEL2ID, POLITIQUE_DECODAGE, construire_fenetres, encoder, predire,
                         WeightedLossTrainer)
from src.extraction import tableau_scores, ecrire_predictions, lire_predictions

VERSION = "APP-v3"
MODELES = {
    "PubMedBERT": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "BioLinkBERT": "michiyasunaga/BioLinkBERT-base",
}
PARAMS = {"learning_rate": 5e-5, "batch_size": 8, "weight_decay": 0.01, "max_length": 512, "entity_weight": 1.0,
          "epochs": 10, "warmup": 0.1}
GRAINES = [42, 43]
GRAINE_FIGEE = 42
TOLERANCE = 0.005

RESULTATS = RACINE / "resultats" / "extraction"
SORTIE = RESULTATS / "essai_rapide" if MODE_RAPIDE else RESULTATS
SORTIE.mkdir(parents=True, exist_ok=True)
if MODE_RAPIDE:
    PARAMS.update(epochs=2, max_length=128)
    GRAINES = [42]

appareil = "GPU " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Appareil :", appareil, "| torch", torch.__version__, "| transformers", transformers.__version__)

Appareil : GPU Tesla T4 | torch 2.11.0+cu130 | transformers 5.20.2


## Étape 1 — Préparer les données (une fois par modèle)

Chaque modèle a son propre vocabulaire : les fenêtres et l'alignement sont recalculés, puis vérifiés comme au 2.4.

In [3]:
documents = charger_jsonl(RACINE / "data" / "corpus_extraction.jsonl")
bio = [json.loads(l) for l in open(RACINE / "data" / "bio_extraction.jsonl", encoding="utf-8")]
textes = {d.document_id: d.texte for d in documents}
bio_par_doc = {b["document_id"]: b for b in bio}
parts = {p: [d for d in documents if d.partition == p] for p in ["train", "validation", "test"]}
if MODE_RAPIDE:
    parts = {"train": parts["train"][:20], "validation": parts["validation"][:10], "test": parts["test"][:10]}

prepa = {}
for nom, chemin in MODELES.items():
    tok = BertTokenizer.from_pretrained(chemin, do_lower_case=True)   # modèles en minuscules
    fen = {p: construire_fenetres([bio_par_doc[d.document_id] for d in docs], textes, tok, PARAMS["max_length"])
           for p, docs in parts.items()}
    don = {p: encoder(fen[p], bio_par_doc, tok, PARAMS["max_length"])[0] for p in ["train", "validation"]}
    for p in ["train", "validation"]:   # même contrôle d'alignement qu'au 2.4
        for f, e in zip(fen[p], don[p]):
            assert [LABELS[l] for l in e["labels"] if l != -100] == [bio_par_doc[f["document_id"]]["tags"][i] for i in f["mots"]]
    prepa[nom] = {"tokenizer": tok, "fenetres": fen, "donnees": don}
    exemple = "Doxorubicin-induced cardiomyopathy"
    print(f"{nom} : fenêtres {dict((p, len(f)) for p, f in fen.items())} | « {exemple} » → {tok.tokenize(exemple)}")
print("BioBERT       : « Doxorubicin-induced cardiomyopathy » →",
      BertTokenizer.from_pretrained("dmis-lab/biobert-base-cased-v1.1").tokenize("Doxorubicin-induced cardiomyopathy"))

PubMedBERT : fenêtres {'train': 518, 'validation': 515, 'test': 513} | « Doxorubicin-induced cardiomyopathy » → ['doxorubicin', '-', 'induced', 'cardiomyopathy']
BioLinkBERT : fenêtres {'train': 518, 'validation': 515, 'test': 511} | « Doxorubicin-induced cardiomyopathy » → ['doxorubicin', '-', 'induced', 'cardiomyopathy']
BioBERT       : « Doxorubicin-induced cardiomyopathy » → ['do', '##x', '##or', '##ubi', '##cin', '-', 'induced', 'card', '##io', '##my', '##op', '##athy']


## Étape 2 — Entraîner et mesurer chaque époque sur VALIDATION

In [4]:
ref_val = mentions_reference(parts["validation"])

def entrainer(nom, graine):
    tok, don = prepa[nom]["tokenizer"], prepa[nom]["donnees"]
    set_seed(graine)
    modele = BertForTokenClassification.from_pretrained(
        MODELES[nom], num_labels=len(LABELS), id2label=dict(enumerate(LABELS)), label2id=LABEL2ID)
    etapes = math.ceil(len(don["train"]) / PARAMS["batch_size"]) * PARAMS["epochs"]
    dossier = DOSSIER_CHECKPOINTS / f"v3_{nom}_graine{graine}"
    args = TrainingArguments(
        output_dir=str(dossier), save_strategy="epoch", save_only_model=True, eval_strategy="no",
        logging_strategy="epoch", learning_rate=PARAMS["learning_rate"], per_device_train_batch_size=PARAMS["batch_size"],
        num_train_epochs=PARAMS["epochs"], weight_decay=PARAMS["weight_decay"], lr_scheduler_type="linear",
        warmup_steps=int(PARAMS["warmup"] * etapes), fp16=torch.cuda.is_available(), report_to="none",
        seed=graine, data_seed=graine, save_total_limit=None)
    trainer = WeightedLossTrainer(model=modele, args=args, entity_weight=PARAMS["entity_weight"],
                                  train_dataset=don["train"], processing_class=tok,
                                  data_collator=DataCollatorForTokenClassification(tok))
    t0 = time.perf_counter()
    trainer.train()
    duree = time.perf_counter() - t0
    points = sorted(dossier.glob("checkpoint-*"), key=lambda c: int(c.name.split("-")[1]))
    return dossier, points, duree, trainer.state.log_history

reglage, historiques = [], {}
for nom in MODELES:
    for graine in GRAINES:
        dossier, points, duree, historique = entrainer(nom, graine)
        historiques[f"{nom}_graine{graine}"] = historique
        for epoque, point in enumerate(points, start=1):
            modele = AutoModelForTokenClassification.from_pretrained(point).to(DEVICE)
            pred = predire(modele, prepa[nom]["tokenizer"], prepa[nom]["fenetres"]["validation"], bio_par_doc,
                           PARAMS["max_length"])
            s = tableau_scores(ref_val, pred, "apprise", "validation").set_index("categorie")
            reglage.append({"modele": nom, "graine": graine, "epoques": epoque, "checkpoint": str(point),
                            "P": s.loc["global (micro)", "precision"], "R": s.loc["global (micro)", "rappel"],
                            "F1": s.loc["global (micro)", "F1"], "F1 Chemical": s.loc["Chemical", "F1"],
                            "F1 Disease": s.loc["Disease", "F1"], "temps_entrainement_s": round(duree, 1)})
            del modele
        if graine != GRAINE_FIGEE:
            shutil.rmtree(dossier)
        print(f"{nom}, graine {graine} : entraînement {duree / 60:.1f} min, "
              f"meilleur F1 {max(r['F1'] for r in reglage if r['modele'] == nom and r['graine'] == graine):.4f}")

reglage = pd.DataFrame(reglage)
reglage.drop(columns="checkpoint").to_csv(SORTIE / "apprise_v3_reglage_validation.csv", index=False)
json.dump(historiques, open(SORTIE / "apprise_v3_historique_entrainement.json", "w"), indent=2, default=str)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING

Step,Training Loss
65,0.613165
130,0.086650
195,0.043009
260,0.022338
325,0.013510
390,0.009202
455,0.004876
520,0.003659
585,0.003096
650,0.002381


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (91 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BioLinkBERT, graine 43 : entraînement 3.1 min, meilleur F1 0.9108


In [5]:
# Référence BioBERT : runs APP-v2 au même taux (5e-5), mêmes graines.
v2 = pd.read_csv(RESULTATS / "apprise_v2_reglage_validation.csv")
v2 = v2[v2.taux == PARAMS["learning_rate"]].assign(modele="BioBERT").drop(columns="taux")
tous = pd.concat([v2, reglage.drop(columns="checkpoint")], ignore_index=True)
moyennes = tous.groupby(["modele", "epoques"])[["F1", "F1 Chemical", "F1 Disease"]].mean().round(4)
moyennes["F1"].unstack("modele")

modele,BioBERT,BioLinkBERT,PubMedBERT
epoques,,,
1,0.8346,0.7931,0.7950
2,0.8792,0.8810,0.8754
3,0.8826,0.8940,0.8916
4,0.8920,0.9013,0.8978
5,0.8919,0.9056,0.9020
6,0.8900,0.9062,0.9050
7,0.8934,0.9018,0.9112
8,0.8952,0.9084,0.9081
9,0.8954,0.9056,0.9092


## Étape 3 — Choisir le modèle

In [6]:
choix = {}
for nom, m in moyennes.groupby(level="modele"):
    m = m.droplevel("modele")
    cand = m[m["F1"] >= m["F1"].max() - TOLERANCE]
    ep = int(cand.index.min())
    choix[nom] = {"epoques": ep, **m.loc[ep].to_dict()}
choix = pd.DataFrame(choix).T
choix["écart à BioBERT"] = (choix["F1"] - choix.loc["BioBERT", "F1"]).round(4)
display(choix)

meilleur = choix.drop(index="BioBERT")["F1"].idxmax()
REMPLACE = choix.loc[meilleur, "écart à BioBERT"] > TOLERANCE
print(f"Meilleur nouveau modèle : {meilleur} ({choix.loc[meilleur, 'F1']:.4f}, écart {choix.loc[meilleur, 'écart à BioBERT']:+.4f})")
print("→ remplace APP-v2" if REMPLACE else "→ gain ≤ 0,005 : APP-v2 (BioBERT) reste la méthode apprise figée")

,epoques,F1,F1 Chemical,F1 Disease,écart à BioBERT
BioBERT,4.0,0.8920,0.9351,0.8380,0.0000
BioLinkBERT,5.0,0.9056,0.9484,0.8514,0.0136
PubMedBERT,7.0,0.9112,0.9514,0.8606,0.0192


Meilleur nouveau modèle : PubMedBERT (0.9112, écart +0.0192)
→ remplace APP-v2


## Étape 4 — Figer le meilleur nouveau modèle, comparer à APP-v2, prédire TEST

Le meilleur nouveau modèle est figé (graine 42) même s'il ne remplace pas APP-v2, pour pouvoir analyser ses
erreurs ; ses prédictions TEST ne serviront dans `5_evaluation` que s'il est retenu.

In [7]:
ep = int(choix.loc[meilleur, "epoques"])
point = reglage[(reglage.modele == meilleur) & (reglage.epoques == ep) & (reglage.graine == GRAINE_FIGEE)].iloc[0]
tok, fen = prepa[meilleur]["tokenizer"], prepa[meilleur]["fenetres"]

DOSSIER_MODELE = RACINE / "modeles" / ("apprise_v3_essai_rapide" if MODE_RAPIDE else "apprise_v3")
if DOSSIER_MODELE.exists():
    shutil.rmtree(DOSSIER_MODELE)
shutil.copytree(point.checkpoint, DOSSIER_MODELE)
modele = AutoModelForTokenClassification.from_pretrained(DOSSIER_MODELE).to(DEVICE)

t0 = time.perf_counter()
pred_val = predire(modele, tok, fen["validation"], bio_par_doc, PARAMS["max_length"])
t_val = time.perf_counter() - t0
scores_val = tableau_scores(ref_val, pred_val, "apprise_v3", "validation", t_val)
scores_val.to_csv(SORTIE / "scores_apprise_v3_validation.csv", index=False)
ecrire_predictions(pred_val, "apprise_v3", parts["validation"], SORTIE / "predictions_apprise_v3_validation.jsonl")

docs_val = {d.document_id for d in parts["validation"]}
pred_v2 = {m for m in lire_predictions(RESULTATS / "predictions_apprise_v2_validation.jsonl") if m.doc in docs_val}
comparaison = pd.concat([tableau_scores(ref_val, pred_v2, "apprise_v2", "validation"), scores_val], ignore_index=True)
comparaison.to_csv(SORTIE / "apprise_v3_comparaison_v2_validation.csv", index=False)
display(comparaison)
pd.DataFrame({n: {k: len(v) for k, v in erreurs(ref_val, p).items()}
              for n, p in [("APP-v2 (BioBERT)", pred_v2), (f"APP-v3 ({meilleur})", pred_val)]}).T

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/tmp/ipykernel_3127/3111566371.py:20: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  comparaison = pd.concat([tableau_scores(ref_val, pred_v2, "apprise_v2", "validation"), scores_val], ignore_index=True)


,methode,partition,categorie,VP,FP,FN,precision,rappel,F1,denominateur_nul,temps_s
0,apprise_v2,validation,Chemical,5018,360,329,0.9331,0.9385,0.9358,False,NaN
1,apprise_v2,validation,Disease,3588,719,656,0.8331,0.8454,0.8392,False,NaN
2,apprise_v2,validation,global (micro),8606,1079,985,0.8886,0.8973,0.8929,False,NaN
3,apprise_v3,validation,Chemical,5141,299,206,0.9450,0.9615,0.9532,False,NaN
4,apprise_v3,validation,Disease,3680,636,564,0.8526,0.8671,0.8598,False,NaN
5,apprise_v3,validation,global (micro),8821,935,770,0.9042,0.9197,0.9119,False,16.35


,correcte,frontiere,type,manquee,parasite
APP-v2 (BioBERT),8606,489,14,482,576
APP-v3 (PubMedBERT),8821,432,4,334,499


In [8]:
t0 = time.perf_counter()
pred_test = predire(modele, tok, fen["test"], bio_par_doc, PARAMS["max_length"])
t_test = time.perf_counter() - t0
ecrire_predictions(pred_test, "apprise_v3", parts["test"], SORTIE / "predictions_apprise_v3_test.jsonl")

gel = {"version": VERSION, "modeles_compares": MODELES, "parametres": PARAMS, "graines": GRAINES,
       "choix_par_modele": choix.round(4).to_dict(orient="index"),
       "modele_fige": {"nom": meilleur, "chemin": MODELES[meilleur], "epochs": ep, "graine": GRAINE_FIGEE},
       "remplace_app_v2": bool(REMPLACE),
       "regle": f"époques : meilleure moyenne, tolérance {TOLERANCE}, moins d'époques ; remplacement si gain > {TOLERANCE}",
       "politique_decodage": POLITIQUE_DECODAGE, "tokenizer": "BertTokenizer, do_lower_case=True",
       "appareil": appareil, "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
       "temps_s": {"entrainement_10_epoques": float(point.temps_entrainement_s), "prediction_validation": round(t_val, 1),
                   "prediction_test": round(t_test, 1)},
       "predictions_test": len(pred_test), "mode_rapide": MODE_RAPIDE}
(SORTIE / "apprise_v3_config_gelee.json").write_text(json.dumps(gel, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(gel, ensure_ascii=False, indent=2))

{
  "version": "APP-v3",
  "modeles_compares": {
    "PubMedBERT": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
    "BioLinkBERT": "michiyasunaga/BioLinkBERT-base"
  },
  "parametres": {
    "learning_rate": 5e-05,
    "batch_size": 8,
    "weight_decay": 0.01,
    "max_length": 512,
    "entity_weight": 1.0,
    "epochs": 10,
    "warmup": 0.1
  },
  "graines": [
    42,
    43
  ],
  "choix_par_modele": {
    "BioBERT": {
      "epoques": 4.0,
      "F1": 0.892,
      "F1 Chemical": 0.9351,
      "F1 Disease": 0.838,
      "écart à BioBERT": 0.0
    },
    "BioLinkBERT": {
      "epoques": 5.0,
      "F1": 0.9056,
      "F1 Chemical": 0.9484,
      "F1 Disease": 0.8514,
      "écart à BioBERT": 0.0136
    },
    "PubMedBERT": {
      "epoques": 7.0,
      "F1": 0.9112,
      "F1 Chemical": 0.9514,
      "F1 Disease": 0.8606,
      "écart à BioBERT": 0.0192
    }
  },
  "modele_fige": {
    "nom": "PubMedBERT",
    "chemin": "microsoft/BiomedNLP-BiomedBERT-base-unc